 Read the bronzedata,create dataframe

In [1]:
customers_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/customers.parquet")
orders_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/orders.parquet")
payments_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/payments.parquet")
support_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/support_tickets.parquet")
web_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/Bronze/web_activities.parquet")


StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 3, Finished, Available, Finished, False)

In [3]:
display(customers_raw.limit(5))

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5ec940dc-2385-4f08-ba90-99f0686025e6)

In [4]:
display(orders_raw.limit(5))

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e9fe560c-05c1-4a98-a7ab-2764cde19fe7)

In [5]:
display(payments_raw.limit(5))

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, febd9020-6bad-491c-9cd3-557217933d8a)

In [7]:
display(support_raw.limit(5))

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 22843d4d-cab0-4a7a-a171-21ff29d11c85)

In [8]:
display(web_raw.limit(5))

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e94ad9c7-2123-4316-8b85-e8ccd92a5ff7)

create deltabronze table

In [9]:

# Save as Bronze Delta Tables
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_raw.write.format("delta").mode("overwrite").saveAsTable("support")
web_raw.write.format("delta").mode("overwrite").saveAsTable("web")

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 11, Finished, Available, Finished, False)

create the data-silver

customers data cleaning

In [10]:
display(customers_raw.limit(5))

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bc6c3b45-e6db-44c6-9522-9d9e09b9cf9e)

In [12]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


customers_clean = (
    customers_raw
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                          .when(lower(col("gender")).isin("m", "male"), "Male")
                          .otherwise("Other"))
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)
display(customers_clean.limit(6))
customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")


StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c5c45b5-094c-4727-9f84-efe3acb7af79)

clean orders table

In [14]:
display(orders_raw.limit(6))

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8138b7bb-8372-497d-a798-58042534aa94)

In [17]:
# Clean orders
orders = spark.table("orders")
orders_clean = (
    orders
    .withColumn("order_date", 
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
display(orders_clean.limit(5))
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")


StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf1f1d0e-1771-4018-a89f-e3ffb697ae15)

In [19]:
# Clean payments
payments = spark.table("payments")
payments_clean = (
    payments
    .withColumn("payment_date", to_date(regexp_replace(col("payment_date"), "/", "-")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
display(payments_clean.limit(5))
payments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")


StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6201cab2-9eba-4286-b5ec-6ebea3bbccd6)

In [21]:
# Clean support
support = spark.table("support")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)
display(support_clean.limit(5))
support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9ae9403a-518d-4e30-be82-b5380a03f250)

In [31]:
%%sql
select *from web

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 33, Finished, Available, Finished, False)

<Spark SQL result set with 15 rows and 5 fields>

In [24]:

# Clean web
web = spark.table("web")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean.limit(5))
web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cbc187f8-2f02-4e7d-bc3d-1fab31888567)

gold tables -aggregate tables

In [30]:
from pyspark.sql.functions import col

cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select(
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status"),

        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)

display(customer360.limit(10))

customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, 3643db5f-3a15-411c-b6b1-a34cadac7117, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 87037437-857a-40fd-a16b-4536d2ca0479)